# Lab 10: Email Spam and Phishing Detection

**Task:** detect spam (and phishing-like) emails.

**Dataset:** Enron-Spam, built from the Enron email corpus (https://www.cs.cmu.edu/~enron/). The raw Enron corpus has no spam labels, so we use the labelled Enron-Spam collection (Metsis, Androutsopoulos and Paliouras, 2006: real ham from six Enron employees' mailboxes mixed with real spam), in the CSV version from https://github.com/MWiechmann/enron_spam_data, included here as `data/enron_spam_data.zip`. Each email has a subject, a message body, a date and a ham/spam label. (This version keeps only the subject line from the headers; there is no From field.)

**What we will do:**
1. Build the dataset: spam vs ham, and a "phishy-looking" subset found with keyword rules.
2. Preprocess: header features from the subject line; TF-IDF on the body.
3. Train multinomial Naive Bayes and logistic regression, with and without the header features.
4. Evaluate with precision, recall, F1 and ROC AUC.
5. Show the strongest spam cues, and analyse 10 false positives and 10 false negatives.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 150)
pd.set_option("display.max_colwidth", 110)

emails = pd.read_csv("data/enron_spam_data.zip")
print(emails.shape)
print(emails["Spam/Ham"].value_counts().to_dict())
emails.head(3)

## 1. Building the dataset

### Cleaning
- 289 emails have no subject and 371 no body: those are empty strings, not reasons to drop an email (spam with an empty subject is still spam).
- **Duplicates.** The same spam is often sent thousands of times. If one copy lands in the training set and another in the test set, the test only checks memory. We keep one copy of each (subject, body) pair.

In [ ]:
emails["Subject"] = emails["Subject"].fillna("")
emails["Message"] = emails["Message"].fillna("")
before = len(emails)
emails = emails.drop_duplicates(subset=["Subject", "Message"]).reset_index(drop=True)
emails["spam"] = (emails["Spam/Ham"] == "spam").astype(int)
print(f"removed {before - len(emails)} duplicates; {len(emails)} emails left, {emails['spam'].mean():.1%} spam")

The text was already lowercased and tokenised by the dataset's authors (punctuation separated by spaces), so "phishy" cues such as URLs appear as "http : / / ...".

### A phishy-looking subset
Phishing emails try to get the reader to log in, verify an account or give out personal or bank details. With no labels for phishing, we can find *candidates* with keyword rules: emails that mention an account or password together with a request to verify, update or confirm, or a link.

In [ ]:
account_words = r"\b(account|password|login|log in|bank|paypal|ebay|credit card|social security|pin)\b"
action_words = r"\b(verify|verification|confirm|update|suspend|suspended|unlock|validate|click here|login)\b"
text_all = emails["Subject"] + " " + emails["Message"]
emails["phishy"] = (text_all.str.contains(account_words) & text_all.str.contains(action_words)).astype(int)
print(pd.crosstab(emails["phishy"], emails["spam"], margins=True, rownames=["phishy rule"], colnames=["spam"]))
for _, r in emails[(emails["phishy"] == 1) & (emails["spam"] == 1)].sample(3, random_state=1).iterrows():
    print("-", r["Subject"][:80], "|", r["Message"][:150].replace("\n", " "))

Keyword rules are crude: they catch genuine phishing-style spam, but also legitimate Enron emails about passwords and account updates (the "phishy" ham). This is why we learn from labelled data instead of writing rules. We will see later how well the spam model scores the phishy subset.

## 2. Preprocessing

### Header features (from the subject line)
Spammers' subject lines often look different: shouting in capitals, many exclamation marks or `$` signs, numbers, missing subjects, or fake replies ("re :"). We turn the subject into a few numeric features. Since the subjects are already lowercased, we can't measure capital letters, but we can measure the rest.

In [ ]:
def header_features(df):
    s = df["Subject"]
    return pd.DataFrame({
        "subj_length": s.str.len(),
        "subj_words": s.str.split().str.len(),
        "subj_empty": (s.str.strip() == "").astype(int),
        "subj_exclaim": s.str.count("!"),
        "subj_dollar": s.str.count(r"\$"),
        "subj_digits": s.str.count(r"\d"),
        "subj_is_reply": s.str.startswith(("re :", "re:")).astype(int),
        "subj_is_forward": s.str.startswith(("fw :", "fwd :", "fw:", "fwd:")).astype(int),
        "body_length_log": np.log1p(df["Message"].str.len()),
        "body_has_link": df["Message"].str.contains(r"http|www").astype(int),
    }, index=df.index)

H = header_features(emails)
H.groupby(emails["spam"]).mean().T.round(2).rename(columns={0: "ham (mean)", 1: "spam (mean)"})

Clear differences: ham is much more often a reply or forward (people answering each other), spam has more links, digits and `$` signs.

### TF-IDF on the body
Each email body becomes a vector of TF-IDF weights: how often each word appears in the email, down-weighted for words common to most emails, and scaled to length 1. We include the subject words too (they're part of the text), use unigrams and bigrams, and ignore terms in fewer than 3 emails.

In [ ]:
from sklearn.model_selection import train_test_split

emails["text"] = emails["Subject"] + " " + emails["Message"]
data = pd.concat([emails[["text", "spam", "phishy", "Subject", "Message"]], H], axis=1)
train, test = train_test_split(data, test_size=0.25, stratify=data["spam"], random_state=0)
header_cols = H.columns.tolist()
print(len(train), "training emails,", len(test), "test emails")

## 3. Models

Naive Bayes needs non-negative features (they are treated like counts), so the header features are scaled to the range 0–1 with a `MinMaxScaler`; for logistic regression we standardise them. Everything is learned from the training data only, inside a pipeline.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

def build(model_name, with_headers):
    tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9, sublinear_tf=True)
    parts = [("text", tfidf, "text")]
    if with_headers:
        parts.append(("headers", MinMaxScaler() if model_name == "naive bayes" else StandardScaler(), header_cols))
    model = MultinomialNB(alpha=0.1) if model_name == "naive bayes" else LogisticRegression(C=10, max_iter=3000)
    return Pipeline([("features", ColumnTransformer(parts)), ("model", model)])

results, fitted = [], {}
for name in ["naive bayes", "logistic regression"]:
    for with_headers in [False, True]:
        pipe = build(name, with_headers).fit(train, train["spam"])
        p = pipe.predict_proba(test)[:, 1]
        pred = (p >= 0.5).astype(int)
        label = f"{name}{' + header features' if with_headers else ''}"
        fitted[label] = (pipe, p)
        results.append({"model": label, "precision": precision_score(test["spam"], pred), "recall": recall_score(test["spam"], pred),
                        "F1": f1_score(test["spam"], pred), "ROC AUC": roc_auc_score(test["spam"], p)})
pd.DataFrame(results).round(4)

## 4. Evaluation

All four models are very accurate (F1 about 0.99, ROC AUC 0.9995): spam vs ham is one of the easiest text classification problems, because spam vocabulary is so different from business email (but see the warnings in section 5). The header features add almost nothing (logistic regression F1 0.9907 → 0.9914; Naive Bayes slightly worse): almost everything they capture (links, money, replies) is already visible in the words. Logistic regression and Naive Bayes are practically tied.

How does the best model do on the phishy-looking subset?

In [ ]:
best = max(fitted, key=lambda n: roc_auc_score(test["spam"], fitted[n][1]))
pipe, p = fitted[best]
print("best:", best)
test = test.assign(p_spam=p, pred=(p >= 0.5).astype(int))
for flag, group in test.groupby("phishy"):
    print(f"phishy rule = {flag}: {len(group)} emails, accuracy {(group['pred'] == group['spam']).mean():.3f}, "
          f"recall on spam {recall_score(group['spam'], group['pred']):.3f}, false alarms on ham {((group['pred'] == 1) & (group['spam'] == 0)).sum()}")
print(pd.DataFrame(confusion_matrix(test["spam"], test["pred"]), index=["ham", "spam"], columns=["predicted ham", "predicted spam"]))

## 5. Deliverables

### Top spam and ham cues (logistic regression weights)

In [ ]:
lr_pipe = fitted["logistic regression + header features"][0]
names = [n.split("__", 1)[1] for n in lr_pipe[0].get_feature_names_out()]
w = pd.Series(lr_pipe[-1].coef_[0], index=names)
pd.DataFrame({"spam cue": w.nlargest(25).index, "weight": w.nlargest(25).values.round(2),
              "ham cue": w.nsmallest(25).index, "weight ": w.nsmallest(25).values.round(2)})

**Spam cues** are marketing and money words ("your", "money", "free", "software", "online", "mobile", "remove", "click here", "viagra"), and an empty subject line (`subj_empty`, one of our header features). **Ham cues** are the language of Enron's business: "enron", colleagues' first names ("vince", "louise"), "713" (Houston's phone area code), "houston", "gas", "power", "energy", "meeting", "deal", "attached", "thanks".

**Two warnings hidden in this list:**
1. **Years are among the strongest cues**: "2004" and "2005" point to spam, "2000" and "2001" to ham. That is not a property of spam: the ham comes from Enron mailboxes of 2000–2002 and the spam was collected later. The model has partly learned *when* an email was written. This is a form of leakage that comes from how the dataset was assembled; a filter deployed in 2006 would find "2006" in both spam and ham.
2. Much of the ham signal is *Enron-specific* (names, Houston, energy trading). The model learned "Enron email vs spam" as much as "legitimate email vs spam". On another company's mail these cues would not transfer, and the model would need retraining. Accuracy figures above 99% should be read with both points in mind.

### 10 false positives (ham flagged as spam)

In [ ]:
fp = test[(test["spam"] == 0) & (test["pred"] == 1)].sort_values("p_spam", ascending=False)
print(len(fp), "false positives")
for _, r in fp.head(10).iterrows():
    print(f"p={r['p_spam']:.2f} | subject: {r['Subject'][:60]!r} | {r['Message'][:150]!r}")

### 10 false negatives (spam that got through)

In [ ]:
fn = test[(test["spam"] == 1) & (test["pred"] == 0)].sort_values("p_spam")
print(len(fn), "false negatives")
for _, r in fn.head(10).iterrows():
    print(f"p={r['p_spam']:.2f} | subject: {r['Subject'][:60]!r} | {r['Message'][:150]!r}")

### Analysis

**False positives** (legitimate mail flagged as spam): almost all are *personal* emails, not business ones: jokes and chain letters forwarded between friends ("wassup!!!!!!", a red-neck valentine poem, "women really important to read"), a forwarded car-rental advert, a note about a new free-email address with its advertising footer. They contain exclamation marks, informal language and marketing text, and none of Enron's business vocabulary, so to the model they look like spam.

**False negatives** (spam that got through) fall into a few groups:
- **business-looking spam**: short messages about invoices, a service cancellation, "please read the attached file" (the payload is in an attachment the model never sees);
- **word-salad spam**: random words ("ogle irreconcilablehint baud midscaleclutch ...") inserted precisely to confuse word-based filters by diluting the spam words;
- **spam in other languages** (a Russian event listing), whose words barely occur in training;
- **mislabelled email**: a reply about research on spam filtering ("ling-spam for discourse analysis") is labelled spam in the data, apparently by mistake.

For a real filter, the two errors are not equally bad: a legitimate email lost in the spam folder (false positive) is usually worse than one more spam in the inbox. So we would raise the cutoff above 0.5 to cut false positives, at the cost of letting a little more spam through: the precision–recall trade-off.